# Multi-Step Workflow (Exercise · Starter)


In this exercise, you’ll build a multi-step workflow using LCEL to solve a more complex task than simply generating a joke. 

**Challenge**

Create an AI Business Advisor that:

1. Accepts an industry as input.
2. Generates a business idea.
3. Analyzes the strengths and weaknesses.
4. Formats the results as a final report.


## 0. Import the necessary libs

In [55]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from pydantic import BaseModel, Field

## 1. Instantiate Chat Model with your API Key

To be able to connect with OpenAI, you need to instantiate an ChatOpenAI client passing your OpenAI key.

You can pass the `api_key` argument directly.
```python
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key=os.getenv("OPENAI_API_KEY"),
)
```

In [56]:
# TODO - Instantiate your chat model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    base_url = 'https://openai.vocareum.com/v1',
    api_key = os.getenv("OPENAI_API_KEY")
)

## 2. Your first Chain

In the end of each chain, you should parse the output and save the logs

In [57]:
logs = []

In [58]:
parser = StrOutputParser()

In [59]:
parse_and_log_output_chain = RunnableParallel(
    output=parser, 
    log=RunnableLambda(lambda x: logs.append(x))
)

#print(logs)

## 3. Idea Generation

Craft a prompt to generate a business idea for the given industry. 

Make sure {industry} placeholder is inside your template, so it can be filled when the chain is invoked.

In [60]:
# TODO - Your prompt Template
idea_prompt = PromptTemplate(
    template=(
        "You are an expert business advisor. "
        "Generate one innovative and disruptive business in the industry "
        "{industry}. "
        "Provide a brief description of the idea."
    )
)

In [61]:
# TODO - Create your idea_chain: idea_prompt -> llm -> parse_and_log_output_chain
idea_chain = (idea_prompt | llm | parse_and_log_output_chain)

In [62]:
# TODO - Test your idea_chain invoking it by passing an industy like "agro" to it
idea_result = idea_chain.invoke('fintech')

In [63]:
idea_result["output"]

'**Business Idea: Decentralized Financial Identity (DFI) Platform**\n\n**Description:**\n\nThe Decentralized Financial Identity (DFI) Platform is an innovative fintech solution that leverages blockchain technology to create a secure, user-controlled digital identity for individuals and businesses. This platform aims to revolutionize how financial institutions verify identities, assess creditworthiness, and manage customer relationships.\n\n**Key Features:**\n\n1. **Self-Sovereign Identity (SSI):** Users can create and manage their digital identities without relying on centralized authorities. They control their personal data and can selectively share it with financial institutions, reducing the risk of data breaches and identity theft.\n\n2. **Smart Contracts for Credit Scoring:** The platform utilizes smart contracts to automate the credit scoring process. Users can provide verifiable data (e.g., income, employment history) directly from trusted sources, allowing for real-time credit 

In [64]:
for message in logs:
    print(message.content)

**Business Idea: Decentralized Financial Identity (DFI) Platform**

**Description:**

The Decentralized Financial Identity (DFI) Platform is an innovative fintech solution that leverages blockchain technology to create a secure, user-controlled digital identity for individuals and businesses. This platform aims to revolutionize how financial institutions verify identities, assess creditworthiness, and manage customer relationships.

**Key Features:**

1. **Self-Sovereign Identity (SSI):** Users can create and manage their digital identities without relying on centralized authorities. They control their personal data and can selectively share it with financial institutions, reducing the risk of data breaches and identity theft.

2. **Smart Contracts for Credit Scoring:** The platform utilizes smart contracts to automate the credit scoring process. Users can provide verifiable data (e.g., income, employment history) directly from trusted sources, allowing for real-time credit assessments

## 4. Idea Analysis

Craft a prompt to analyze the generated idea's strengths and weaknesses

In [65]:
# TODO - Your prompt Template
analysis_prompt = PromptTemplate(
    template=(
        "You are an expert software architect, analyze the following idea: "
        "Idea: {idea} "
        "Identify 3 key strengths and weaknesses of the idea and justify with claims."
    )
)

In [66]:
# TODO - Your chain
analysis_chain = (
    analysis_prompt
    | llm
    | parse_and_log_output_chain
)

In [67]:
# TODO - Test your analysis_chain invoking it by passing idea_result["output"] to it
analysis_result = analysis_chain.invoke('MongoDB aggregation library, currently doing aggregations in mongo is a nightmare for the DX')

In [68]:
analysis_result["output"]

"### Key Strengths\n\n1. **Improved Developer Experience (DX)**:\n   - **Justification**: MongoDB's aggregation framework can be complex and unintuitive, especially for developers who are not familiar with its syntax and structure. By creating a dedicated aggregation library, you can abstract away the complexity and provide a more user-friendly interface. This can lead to faster development cycles, reduced onboarding time for new developers, and fewer errors in aggregation queries.\n\n2. **Enhanced Readability and Maintainability**:\n   - **Justification**: A well-designed aggregation library can provide a more declarative and readable way to express data transformations. By using a fluent API or a more intuitive syntax, developers can write aggregation queries that are easier to understand at a glance. This can improve code maintainability, as future developers (or even the original authors) can quickly grasp the intent of the code without needing to decipher complex MongoDB aggregati

In [69]:
for message in logs:
    print(message.content)

**Business Idea: Decentralized Financial Identity (DFI) Platform**

**Description:**

The Decentralized Financial Identity (DFI) Platform is an innovative fintech solution that leverages blockchain technology to create a secure, user-controlled digital identity for individuals and businesses. This platform aims to revolutionize how financial institutions verify identities, assess creditworthiness, and manage customer relationships.

**Key Features:**

1. **Self-Sovereign Identity (SSI):** Users can create and manage their digital identities without relying on centralized authorities. They control their personal data and can selectively share it with financial institutions, reducing the risk of data breaches and identity theft.

2. **Smart Contracts for Credit Scoring:** The platform utilizes smart contracts to automate the credit scoring process. Users can provide verifiable data (e.g., income, employment history) directly from trusted sources, allowing for real-time credit assessments

## 5. Report Generation

Craft a prompt to structure the information into a business report.

In [70]:
# TODO - Your prompt Template
report_prompt = PromptTemplate(
    template=(
        "You are an expert analyst, here you have a task: "
        "Strengths and weaknesses: {output}"
        "Generate a structured business report."
    )
)

In [71]:
class AnalysisReport(BaseModel):
    """Strengths and Weaknesses about a business idea"""
    strengths: list = Field(default=[], description="Idea's strength list")
    weaknesses: list = Field(default=[], description="Idea's weaknesse list")

In [72]:
report_chain = (
    report_prompt | llm.with_structured_output(AnalysisReport, method="function_calling")
)

In [ ]:
# TODO - Test your report_chain invoking it by passing analysis_result["output"] to it
report_result = report_chain.invoke(analysis_result["output"])

In [ ]:
report_result
print(report_result.model_dump_json(indent=2))

## 6. End to End Chain

In [ ]:
e2e_chain = ( 
    RunnablePassthrough() 
    | idea_chain
    | RunnableParallel(idea=RunnablePassthrough())
    | analysis_chain
    | report_chain
)

In [ ]:
e2e_chain.get_graph().print_ascii()

In [ ]:
# Change the industry if you want
e2e_result = e2e_chain.invoke("ai")

In [ ]:
e2e_result

In [46]:
e2e_result.strengths

['Transparency and Traceability: The use of blockchain technology ensures that every transaction is recorded on a decentralized ledger, providing complete traceability of products from farm to consumer, enhancing consumer trust.',
 'Direct-to-Consumer Sales Model: By allowing farmers to sell directly to consumers, SABM eliminates intermediaries, leading to higher profit margins for farmers and lower prices for consumers, fostering community engagement and loyalty.',
 "Data-Driven Insights: The platform's ability to collect and analyze data on consumer preferences and market trends provides valuable insights to farmers, helping them make informed decisions about crop selection and timing, ultimately leading to better yields and profitability."]

In [47]:
e2e_result.weaknesses

['Technology Adoption Barriers: Significant barriers to adoption may exist, particularly among smallholder farmers who may lack the technical knowledge or resources to engage with the platform effectively.',
 'Regulatory and Compliance Challenges: The agricultural sector is heavily regulated, and navigating the legal landscape can be complex, requiring compliance with various regulations that could hinder growth if not managed properly.',
 'Market Competition and Scalability: The agricultural marketplace is competitive, and SABM will need to differentiate itself while effectively marketing its value proposition and scaling the platform to accommodate a growing user base.']

In [48]:
logs

[AIMessage(content='**Business Idea: Decentralized Financial Identity (DFI) Platform**\n\n**Description:**\n\nThe Decentralized Financial Identity (DFI) Platform is an innovative fintech solution that leverages blockchain technology to create a secure, user-controlled digital identity for individuals and businesses. This platform aims to revolutionize how financial institutions verify identities, assess creditworthiness, and provide personalized financial services.\n\n**Key Features:**\n\n1. **Self-Sovereign Identity (SSI):** Users can create and manage their digital identities without relying on centralized authorities. They control their personal data and can selectively share it with financial institutions, reducing the risk of data breaches and identity theft.\n\n2. **Smart Contracts for Credit Scoring:** The platform utilizes smart contracts to automate the credit scoring process. Users can link their financial behaviors (like savings, investments, and payment histories) directly 

## 7. Experiment

Now that you understood how it works, experiment with new things.
- Improve memory
- Explore the Runnables
- Add More Complexity